# Lezione 08 - Pattern di Progettazione Multi-Agent


## Configurazione


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Perché i Sistemi Multi-Agente?

Compiti del mondo reale come la pianificazione di un viaggio coinvolgono molti tipi diversi di competenze — logistica, conoscenza locale, budget e altro. Un singolo agente che cerca di gestire tutto diventa rapidamente ingestibile.

I sistemi multi-agente risolvono questo tramite la **specializzazione**: ogni agente si concentra su un'area di competenza, producendo risultati di qualità superiore rispetto a un generalista. Migliorano anche la **scalabilità** — puoi aggiungere nuovi agenti (ad esempio, uno specialista di voli, un critico gastronomico) senza riscrivere il flusso di lavoro esistente. Gli agenti si compongono insieme attraverso una pipeline strutturata, passando il contesto da uno all'altro.


## Creazione di Agenti Specializzati


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Costruire un Workflow Sequenziale

`WorkflowBuilder` ti permette di collegare agenti in un grafo diretto. Qui creiamo una pipeline semplice in due passaggi: il **TravelPlanner** stila l'itinerario, poi il **TravelConcierge** lo rivede e lo migliora.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Aggiungere più agenti al workflow

Uno dei maggiori vantaggi del modello multi-agente è la facilità con cui può essere esteso. Di seguito aggiungiamo un agente **BudgetReviewer** che controlla il piano rispetto al budget del viaggiatore, segnala gli elementi che potrebbero far superare il limite dei costi e suggerisce alternative per risparmiare denaro. Ora il workflow esegue tre agenti in sequenza:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Riepilogo

In questa lezione hai imparato a:

1. **Creare agenti specializzati** — ognuno con un ruolo focalizzato (pianificazione, concierge, revisione del budget).
2. **Collegare gli agenti in un flusso di lavoro sequenziale** usando `WorkflowBuilder` e `add_edge`.
3. **Trasmettere in streaming l’output** di una pipeline multi-agente, tracciando quale agente sta parlando.
4. **Estendere un flusso di lavoro** aggiungendo nuovi agenti alla catena senza modificare quelli esistenti.

Il modello di progettazione multi-agente mantiene ogni agente semplice, producendo risultati più ricchi e più attentamente revisionati di quelli che un singolo agente potrebbe ottenere da solo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
